# Ensemble Methods Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Decision Stump (Base Learner)

The code in `code/ensembles.py` implements everything from scratch. We start with a decision stump: a tree with a single split.

In [ ]:
```python

class DecisionStump:

    def __init__(self):

        self.feature_idx = None

        self.threshold = None

        self.polarity = 1

        self.alpha = None

    def fit(self, X, y, weights):

        n_samples, n_features = X.shape

        best_error = float("inf")

        for f in range(n_features):

            thresholds = np.unique(X[:, f])

            for thresh in thresholds:

                for polarity in [1, -1]:

                    pred = np.ones(n_samples)

                    pred[polarity * X[:, f] < polarity * thresh] = -1

                    error = np.sum(weights[pred != y])

                    if error < best_error:

                        best_error = error

                        self.feature_idx = f

                        self.threshold = thresh

                        self.polarity = polarity

    def predict(self, X):

        n = X.shape[0]

        pred = np.ones(n)

        idx = self.polarity * X[:, self.feature_idx] < self.polarity * self.threshold

        pred[idx] = -1

        return pred

In [ ]:
```

### Step 2: AdaBoost from Scratch

In [ ]:
```python

class AdaBoostScratch:

    def __init__(self, n_estimators=50):

        self.n_estimators = n_estimators

        self.stumps = []

        self.alphas = []

    def fit(self, X, y):

        n = X.shape[0]

        weights = np.full(n, 1 / n)

        for _ in range(self.n_estimators):

            stump = DecisionStump()

            stump.fit(X, y, weights)

            pred = stump.predict(X)

            err = np.sum(weights[pred != y])

            err = np.clip(err, 1e-10, 1 - 1e-10)

            alpha = 0.5 * np.log((1 - err) / err)

            weights *= np.exp(-alpha * y * pred)

            weights /= weights.sum()

            stump.alpha = alpha

            self.stumps.append(stump)

            self.alphas.append(alpha)

    def predict(self, X):

        total = sum(a * s.predict(X) for a, s in zip(self.alphas, self.stumps))

        return np.sign(total)

In [ ]:
```

### Step 3: Gradient Boosting from Scratch

In [ ]:
```python

class GradientBoostingScratch:

    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):

        self.n_estimators = n_estimators

        self.lr = learning_rate

        self.max_depth = max_depth

        self.trees = []

        self.initial_pred = None

    def fit(self, X, y):

        self.initial_pred = np.mean(y)

        current_pred = np.full(len(y), self.initial_pred)

        for _ in range(self.n_estimators):

            residuals = y - current_pred

            tree = SimpleRegressionTree(max_depth=self.max_depth)

            tree.fit(X, residuals)

            update = tree.predict(X)

            current_pred += self.lr * update

            self.trees.append(tree)

    def predict(self, X):

        pred = np.full(X.shape[0], self.initial_pred)

        for tree in self.trees:

            pred += self.lr * tree.predict(X)

        return pred

In [ ]:
```

### Step 4: Compare against sklearn

The code verifies that our from-scratch implementations produce similar accuracy to sklearn's `AdaBoostClassifier` and `GradientBoostingClassifier`, and compares all methods side by side.

## Exercises

In [ ]:
1. Modify the AdaBoost implementation to track training accuracy after each round. Plot accuracy vs. number of estimators. When does it converge?

2. Implement a random forest from scratch by adding random feature subsampling to the regression tree. Train 100 trees with `max_features=sqrt(n_features)` and average predictions. Compare variance reduction to a single tree.

3. In the gradient boosting implementation, add early stopping: track validation loss after each round and stop when it has not improved for 10 consecutive rounds. How many trees does it actually need?

4. Build a stacking ensemble with three base models (logistic regression, decision tree, k-nearest neighbors) and a logistic regression meta-learner. Use 5-fold cross-validation to generate meta-features. Compare to each base model alone.

5. Run XGBoost on the same dataset with default parameters. Compare its accuracy to your from-scratch gradient boosting. Time both. How large is the speed difference?